# Approach 1 — DenseNet + attention LSTM report generator

This is a small, reproducible image-to-text baseline. DenseNet-121 supplies spatial image features; an additive-attention LSTM predicts report tokens. It is an educational experiment, not a diagnostic tool. Start with a small training subset, then increase it only after the pipeline runs end to end.

## Runtime notes

Use a Colab GPU for practical training. This model downloads ImageNet DenseNet weights the first time. Training defaults to a small subset (`TRAIN_LIMIT=512`) so a first run is manageable. Set it to `None` for all frontal training studies after the smoke run. Validation is for development; do not tune on the test set.

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/cxr_second_reader"), Path("/content/cxr_second_reader")]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "data.py").exists()), None)
if REPO_ROOT is None and Path("/content").exists():
    import subprocess
    clone_dir = Path("/content/cxr_second_reader")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/eyadstein/cxr_second_reader.git", str(clone_dir)], check=True)
    if (clone_dir / "src" / "data.py").exists():
        REPO_ROOT = clone_dir.resolve()
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the cxr_second_reader project checkout, or make the updated project available in Drive/GitHub.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Project:", REPO_ROOT)

In [ ]:
import json, re, math
from functools import partial
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from src.experiment_utils import study_rows, ImageTextRows
from src.report_models import ReportVocabulary, report_collate, AdditiveAttentionLSTM
from src.evaluate import evaluate

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE, EPOCHS, MAX_LENGTH = 4, 1, 128
TRAIN_LIMIT, VAL_LIMIT = 512, 64  # Set either to None for the full split.
print("device:", DEVICE)

In [ ]:
train_df = study_rows("train", TRAIN_LIMIT)
val_df = study_rows("validation", VAL_LIMIT)
print("frontal studies:", len(train_df), len(val_df))
vocab = ReportVocabulary(train_df["target_text"].tolist(), max_size=12000, min_frequency=2)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_loader = DataLoader(
    ImageTextRows(train_df, transform), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=(DEVICE.type == "cuda"),
    collate_fn=partial(report_collate, vocab=vocab, max_length=MAX_LENGTH),
)
val_loader = DataLoader(
    ImageTextRows(val_df, transform), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=(DEVICE.type == "cuda"),
    collate_fn=partial(report_collate, vocab=vocab, max_length=MAX_LENGTH),
)
model = AdditiveAttentionLSTM(len(vocab.to_token), vocab.pad_id, pretrained=True, freeze_encoder=True).to(DEVICE)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
print("vocabulary:", len(vocab.to_token), "trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
for epoch in range(EPOCHS):
    model.train(); losses = []
    for images, token_ids in train_loader:
        images, token_ids = images.to(DEVICE), token_ids.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images, token_ids)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), token_ids[:, 1:].reshape(-1),
            ignore_index=vocab.pad_id,
        )
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss; stop and inspect the batch/runtime.")
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    print(f"epoch {epoch + 1}/{EPOCHS} mean_loss={np.mean(losses):.4f}")

@torch.inference_mode()
def predict_loader(loader, limit=24):
    model.eval(); predictions=[]; references=[]
    for images, token_ids in loader:
        images = images.to(DEVICE)
        generated = model.generate(images, vocab.bos_id, vocab.eos_id, max_new_tokens=MAX_LENGTH-1)
        predictions.extend(vocab.decode(row.tolist()) for row in generated.cpu())
        references.extend(vocab.decode(row.tolist()) for row in token_ids)
        if len(predictions) >= limit: break
    return predictions[:limit], references[:limit]

val_predictions, val_references = predict_loader(val_loader, limit=24)
val_metrics = evaluate(val_predictions, val_references)
print("validation text-overlap metrics (not clinical accuracy):", val_metrics)
OUT = Path("outputs/02_attention_lstm"); OUT.mkdir(parents=True, exist_ok=True)
torch.save({"model": model.state_dict(), "vocab": vocab.to_token, "max_length": MAX_LENGTH}, OUT / "checkpoint.pt")
(OUT / "validation_metrics.json").write_text(json.dumps(val_metrics, indent=2), encoding="utf-8")
print("saved:", OUT.resolve())

## Final test evaluation (run once after selecting settings)

The test split is kept separate from training and checkpoint selection. When ready, replace `val_df` with `study_rows("test", None)`, rebuild its loader with the same transform and collate function, then call `predict_loader` once and save the metrics. Keep the test set out of iterative tuning.